# Predicción de Informalidad Laboral en Colombia: Ingesta de Datos
**Fuente:** Gran Encuesta Integrada de Hogares (GEIH) — DANE 2025  
**Entorno:** Google Colab + Google Drive

---
### SetUp

Conexión con Google Drive, instalación e importe de librerías.

Estructura de archivos dentro de Drive:
```
Mi unidad/
└── GEIH_2025/
    ├── Caracteristicas_Generales/
    │   ├── 1_Caracteristicas_Generales_Enero.csv
    │   ├── 2_Caracteristicas_Generales_Febrero.csv
    │   └── ...
    └── Ocupados/
        ├── 1_Ocupados_Enero.csv
        ├── 2_Ocupados_Febrero.csv
        └── ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install duckdb pyarrow --quiet
import pandas as pd
import duckdb
from pathlib import Path

---
### Extracción de datos

In [ ]:

# Ruta de Google Drive ajustable
# Nombres de carpetas y archivos también ajustables, mientras
# coincidan con los valores de esta sección.

BASE_DRIVE        = Path('/content/drive/MyDrive/GEIH_2025')
CARPETA_GENERALES = BASE_DRIVE / 'Caracteristicas_Generales'
CARPETA_OCUPADOS  = BASE_DRIVE / 'Ocupados'
CARPETA_SALIDA    = BASE_DRIVE / 'Procesado'
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
SEPARADOR = ';'
ENCODING  = 'latin-1'

for carpeta, nombre in [(CARPETA_GENERALES, 'Características Generales'),
                         (CARPETA_OCUPADOS,  'Ocupados')]:
    if carpeta.exists():
        archivos = sorted(carpeta.glob('*.csv'))
        print(f'{nombre}: {len(archivos)} archivos CSV')
        for a in archivos:
            print(f'{a.name}')
    else:
        print(f'Carpeta no encontrada: {carpeta}')

#### Columnas a cargar

Se cargan las columnas consideradas relevantes y suficientes para un volumen de datos adecuado y razonable para el alcance relativamente corto del proyecto. Un solo archivo de Características Generales en su estado natural contiene 55 columnas y, en promedio, 70000 filas. Para un total de casi 4 millones de datos por archivo. 

**Módulo Características Generales:**

| Variable | Descripción |
|---|---|
| `P3271` | Sexo (1=Hombre, 2=Mujer) |
| `P6040` | Edad en años cumplidos |
| `P3042` | Nivel educativo (1=Ninguno a 13=Doctorado) |
| `P3042S1` | Años aprobados en ese nivel |
| `P6070` | Estado civil |
| `DPTO` | Código del departamento (33 dominios DANE) |
| `CLASE` | Zona (1=Cabecera municipal, 2=Rural) |
| `FEX_C18` | Factor de expansión estadística |

**Módulo Ocupados:**

| Variable | Descripción |
|---|---|
| **`P6920`** | **Variable objetivo** (1=Cotizante, 2=No cotizante, 3=Pensionado) |
| `P6430` | Posición ocupacional |
| `P6450` | Tipo de contrato (verbal, escrito, no sabe) |
| `P6460` | Naturaleza del contrato (indefinido, fijo, NS/NR) |
| `P6800` | Horas trabajadas en la semana |
| `P3069` | Tamaño del establecimiento |
| `RAMA2D_R4` | Rama de actividad económica (CIIU 2 dígitos) |
| `P7040` | Pluriempleo (1=Sí, 2=No) |
| `INGLABO` | Ingreso laboral (solo referencia descriptiva) |

In [ ]:
# Llave única que identifica a cada persona en la encuesta
LLAVES = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']


COLS_GENERALES = LLAVES + [
    'P3271',    # Sexo (1=Hombre, 2=Mujer)
    'P6040',    # Edad en años cumplidos
    'P3042',    # Nivel educativo (1=Ninguno a 13=Doctorado)
    'P3042S1',  # Años aprobados en ese nivel educativo
    'P6070',    # Estado civil
    'DPTO',     # Código del departamento (33 dominios DANE)
    'CLASE',    # Zona (1=Cabecera municipal, 2=Rural)
    'FEX_C18',  # Factor de expansión (peso estadístico)
]

COLS_OCUPADOS = LLAVES + [
    'P6920',    # ← VARIABLE OBJETIVO
                #   1 = Cotizante a salud por trabajo      → FORMAL
                #   2 = No cotizante                       → INFORMAL
                #   3 = Pensionado (retirado)
    'P6430',    # Posición ocupacional (empleado, cuenta propia, patrón...)
    'P6450',    # Tipo de contrato (verbal, escrito, no sabe)
    'P6460',    # Naturaleza del contrato (indefinido, fijo, NS/NR)
    'P6800',    # Horas trabajadas en la semana de referencia
    'P3069',    # Tamaño del establecimiento (1 persona a 201 o más)
    'RAMA2D_R4',# Rama de actividad económica (CIIU a 2 dígitos)
    'P7040',    # Pluriempleo (1=Sí, 2=No)
    'INGLABO',  # Ingreso laboral — solo referencia descriptiva, NO predictor
]

print('Columnas definidas correctamente')
print(f' Características Generales: {len(COLS_GENERALES)} columnas')
print(f' Ocupados:                  {len(COLS_OCUPADOS)} columnas')


# Validación de que las columnas escogidas no falten en ninguno de los 
# archivos detectados en el drive

archivos_gen = sorted(CARPETA_GENERALES.glob('*.csv'))
archivos_ocu = sorted(CARPETA_OCUPADOS.glob('*.csv'))


cols_gen = pd.read_csv(archivos_gen[0], nrows=0,
                       encoding=ENCODING, sep=SEPARADOR).columns.tolist()
cols_ocu = pd.read_csv(archivos_ocu[0], nrows=0,
                       encoding=ENCODING, sep=SEPARADOR).columns.tolist()

print(f'Verificando: {archivos_gen[0].name}')
todo_ok = True
for col in COLS_GENERALES:
    ok = col in cols_gen
    if not ok: todo_ok = False
    print(f'  {"OK" if ok else "FALTA"}  {col}')

print(f'\nVerificando: {archivos_ocu[0].name}')
for col in COLS_OCUPADOS:
    ok = col in cols_ocu
    if not ok: todo_ok = False
    print(f'  {"OK" if ok else "FALTA"}  {col}')

if todo_ok:
    print('\nTodas las columnas encontradas.')
else:
    print('\nAlgunas columnas no se encontraron.')


Existe la posibilidad de que año a año cambien los nombres o las descripciones de las variables. Tener en cuenta para futuros reentrenos.

---
### Carga de archivos y unión de módulos

Esta celda puede tardar entre 5 y 15 minutos (Con la volumetría trabajada por nosotros)

In [ ]:
def cargar_modulo(archivos, columnas, nombre):
    dfs = []
    print(f'Cargando: {nombre}')
    for archivo in sorted(archivos):
        
        cols_disponibles = pd.read_csv(
            archivo, nrows=0, encoding=ENCODING, sep=SEPARADOR
        ).columns.tolist()
        cols_a_leer = [c for c in columnas if c in cols_disponibles]

        df_mes = pd.read_csv(
            archivo,
            usecols=cols_a_leer,
            encoding=ENCODING,
            sep=SEPARADOR,
            low_memory=False
        )
        dfs.append(df_mes)
        print(f'  {archivo.name}: {len(df_mes):,} filas')

    resultado = pd.concat(dfs, ignore_index=True)
    print(f'  → Total {nombre}: {len(resultado):,} filas\n')
    return resultado


df_gen = cargar_modulo(archivos_gen, COLS_GENERALES, 'Características Generales')
df_ocu = cargar_modulo(archivos_ocu, COLS_OCUPADOS,  'Ocupados')


df = pd.merge(df_gen, df_ocu, on=LLAVES, how='inner')

print(f'\n=== RESULTADO ===')
print(f'  Características Generales: {len(df_gen):,} personas')
print(f'  Ocupados:                  {len(df_ocu):,} personas')
print(f'  Después del join:          {len(df):,} registros')
print(f'  Columnas:                  {df.shape[1]}')

del df_gen, df_ocu
print('\nMódulos unidos')